# Practice: Does This Corpus Cluster by Topic or by Language?

**Computational Text Analysis Pack · Module 06 — Clustering Documents**

*Technique:* Unsupervised structure, and testing what the clusters actually track

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/06e_practice_clustering.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **When you cluster a multilingual corpus without telling the algorithm
> anything, what does it find — the topics, or the languages?**

The answer is nearly always "the languages", and it matters because clustering
is often presented as discovering themes. Here it discovers orthography.

### How to use this notebook

This is a worked case, not a tutorial. Run it once to see the shape of the
argument, then change the question at the top to one of your own and run it
again. The cells are written so that swapping the question is the only edit you
need.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

sub = df[df["n_words"] >= 30].sample(n=2000, random_state=2).reset_index(drop=True)
vec = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=12000, sublinear_tf=True)
X = vec.fit_transform(sub["text"])
print(f"{X.shape[0]} documents, {X.shape[1]} features, "
      f"{sub['language'].nunique()} languages, {sub['category'].nunique()} categories")

K = sub["language"].nunique()
km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X)
sub["cluster"] = km.labels_
print(f"\nARI against LANGUAGE : {adjusted_rand_score(sub['language'], sub['cluster']):.3f}")
print(f"ARI against CATEGORY : {adjusted_rand_score(sub['category'], sub['cluster']):.3f}")
print()
print("Adjusted Rand Index: 1.0 is perfect agreement, 0.0 is chance.")

In [ ]:
# Which is it? Look inside a few clusters.
for c in sorted(sub["cluster"].unique())[:4]:
    g = sub[sub["cluster"] == c]
    top_lang = g["language"].value_counts()
    top_cat = g["category"].value_counts()
    print(f"cluster {c}  n={len(g)}")
    print(f"   languages : {top_lang.head(3).to_dict()}")
    print(f"   categories: {top_cat.head(3).to_dict()}")
    print()

In [ ]:
# The fix: cluster WITHIN one language, so orthography is held constant.
one = sub[sub["language"] == "English"].reset_index(drop=True)
if len(one) > 60:
    X1 = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=2,
                         max_features=8000, sublinear_tf=True).fit_transform(one["text"])
    k2 = one["category"].nunique()
    km1 = KMeans(n_clusters=k2, random_state=0, n_init=10).fit(X1)
    print(f"within English only ({len(one)} docs, k={k2}):")
    print(f"   ARI against CATEGORY: {adjusted_rand_score(one['category'], km1.labels_):.3f}")
    print()
    print("Higher than before, because the language signal is no longer available")
    print("for the algorithm to find instead of the topics.")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(["all langs\nvs language", "all langs\nvs category", "English only\nvs category"],
       [adjusted_rand_score(sub['language'], sub['cluster']),
        adjusted_rand_score(sub['category'], sub['cluster']),
        adjusted_rand_score(one['category'], km1.labels_) if len(one) > 60 else 0],
       color=["indianred", "grey", "seagreen"])
ax.set_ylabel("adjusted Rand index"); ax.set_title("What the clusters actually track")
plt.tight_layout()

## What this licenses

**Defensible:** "K-means over TF-IDF features on this multilingual corpus
recovers language membership, not topic. Restricting to a single language
recovers topic structure at ARI ≈ [your number]."

**Not defensible:** presenting the first clustering as thematic discovery. It
would be a real error, and an easy one — the clusters are coherent, they look
meaningful, and a reader who cannot see the language labels has no way to
notice.

**The general rule:** an unsupervised method finds the strongest signal in the
representation, which is rarely the signal you were interested in. Always test
your clusters against a variable you already know, exactly as above. If they
track it, your "discovery" was a variable you already had.

### 🔧 Your turn 1

Re-run the multilingual clustering with `analyzer='char_wb'` character n-grams. Does it track language more strongly or less?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

More strongly — character n-grams are close to a pure orthography signal, so the
clusters become almost perfectly aligned with language. That is useful when
language identification *is* the task, and it makes the general point precisely:
the representation determines what the clustering can find. Choosing features is
choosing what you are able to discover, which is why module 6a spends its time
on the document-term matrix rather than on the clustering algorithm.

</details>